<a href="https://colab.research.google.com/github/karnika-soni/Indic-Multimodal-NMT/blob/main/indic_translation_dataset_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT_STR = "/content/drive/MyDrive/Indic-Multimodal-NMT"

PROJECT_ROOT = Path(
    PROJECT_ROOT_STR
)

metadata_path = (
    PROJECT_ROOT_STR + "/data/processed/flickr30k_metadata.csv"
)


In [ ]:
df = pd.read_csv(metadata_path)

df.head()

,image_id,filename,source_text
0,0,1000092795.jpg,Two young guys with shaggy hair look at their ...
1,0,1000092795.jpg,"Two young, White males are outside near many b..."
2,0,1000092795.jpg,Two men in green shirts are standing in a yard.
3,0,1000092795.jpg,A man in a blue shirt standing in a garden.
4,0,1000092795.jpg,Two friends enjoy time spent together.


In [ ]:
!pip install transformers sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.2 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Set source language
tokenizer.src_lang = "en_XX"

text = ["A boy is playing with a dog."]

inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)

outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"],
    max_length=128,
)

translated = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print(translated)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

['एक लड़का कुत्ते के साथ खेल रहा है।']


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
text = [
    "A boy is playing with a dog."
]


inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True
).to(device)

outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"],
    max_length=128,
)

translated = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True
)


translated

['एक लड़का कुत्ते के साथ खेल रहा है।']

In [ ]:
print(df.head())
print(len(df))

   image_id        filename                                        source_text
0         0  1000092795.jpg  Two young guys with shaggy hair look at their ...
1         0  1000092795.jpg  Two young, White males are outside near many b...
2         0  1000092795.jpg    Two men in green shirts are standing in a yard.
3         0  1000092795.jpg        A man in a blue shirt standing in a garden.
4         0  1000092795.jpg             Two friends enjoy time spent together.
155070


In [ ]:
def translate_batch(texts):

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():

        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"],
            max_length=128
        )

    translations = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

    return translations

In [ ]:
sample = df.head(10).copy()

sample["hindi"] = translate_batch(
    sample["source_text"].tolist()
)

sample[
    ["source_text", "hindi"]
]

,source_text,hindi
0,Two young guys with shaggy hair look at their ...,दो बूढ़े-बूढ़े बाल वाले लड़के yard में बैठे-बै...
1,"Two young, White males are outside near many b...",दो युवा सफेद नर बाहर अनेक वृक्षों के पास रहते ...
2,Two men in green shirts are standing in a yard.,हरे-भरे कमर पहने दो आदमी एक आँगन में खड़े हैं।
3,A man in a blue shirt standing in a garden.,एक नीली शरीर पहने हुए आदमी जो एक बाग में खड़ा है।
4,Two friends enjoy time spent together.,दो मित्रों को एक साथ बिताने का आनंद होता है।
5,Several men in hard hats are operating a giant...,भारी टोपी पहने हुए कई लोग एक विशाल पंखा प्रणाल...
6,Workers look down from up above on a piece of ...,मजदूर ऊपर से एक उपकरण पर नीचे देख रहे हैं।
7,Two men working on a machine wearing hard hats.,कठोर टोपी पहने हुए एक मशीन पर काम करने वाले दो...
8,Four men on top of a tall structure.,एक ऊंची इमारत के ऊपर चार आदमी।
9,Three men on a large rig.,एक बड़े यंत्र पर तीन आदमी।


In [ ]:
from pathlib import Path
import pandas as pd
import torch
from tqdm import tqdm


CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

# CHECKPOINT_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )


CHECKPOINT_FILE = CHECKPOINT_DIR / "translation_checkpoint.csv"

print(CHECKPOINT_FILE)

/content/drive/MyDrive/Indic-Multimodal-NMT/checkpoints/translation_checkpoint.csv


In [ ]:
if CHECKPOINT_FILE.exists():

    translated_df = pd.read_csv(
        CHECKPOINT_FILE
    )

    start_index = len(translated_df)

    print(
        f"Resuming from {start_index} samples"
    )

else:

    translated_df = pd.DataFrame(
        columns=[
            "image_id",
            "filename",
            "source_text",
            "target_text"
        ]
    )

    start_index = 0

    print("Starting new translation job")

Resuming from 25840 samples


In [ ]:
BATCH_SIZE = 32

SAVE_EVERY = 20   # batches

# tqdm is used here to display a smart progress bar, visualizing the progress of the batch translation.
# It wraps around any iterable (in this case, the range of indices) and provides real-time updates
# on the number of iterations completed, elapsed time, and estimated time remaining.
for batch_num, start in enumerate(
    tqdm(
        range(
            start_index,
            len(df),
            BATCH_SIZE
        )
    )
):

    batch_df = df.iloc[
        start:start+BATCH_SIZE
    ]


    translations = translate_batch(
        batch_df["source_text"].tolist()
    )


    batch_result = pd.DataFrame(
        {
            "image_id":
                batch_df["image_id"].values,

            "filename":
                batch_df["filename"].values,

            "source_text":
                batch_df["source_text"].values,

            "target_text":
                translations
        }
    )


    translated_df = pd.concat(
        [
            translated_df,
            batch_result
        ],
        ignore_index=True
    )


    # checkpoint save
    if batch_num % SAVE_EVERY == 0:

        translated_df.to_csv(
            CHECKPOINT_FILE,
            index=False
        )

        print(
            f"\nSaved checkpoint at {len(translated_df)} samples"
        )


  0%|          | 1/8118 [01:51<251:08:24, 111.38s/it]


Saved checkpoint at 25200 samples


  0%|          | 11/8118 [17:19<183:46:30, 81.61s/it]


Saved checkpoint at 25360 samples


  0%|          | 21/8118 [31:51<205:07:24, 91.20s/it]


Saved checkpoint at 25520 samples


  0%|          | 31/8118 [47:04<191:48:24, 85.38s/it]


Saved checkpoint at 25680 samples


  0%|          | 40/8118 [59:58<192:34:48, 85.82s/it]

In [ ]:
import sys
import torch
import sympy
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("SymPy:", sympy.__version__)
print("Transformers:", transformers.__version__)
print("SymPy location:", sympy.__file__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cpu
SymPy: 1.14.0
Transformers: 5.13.1
SymPy location: /usr/local/lib/python3.12/dist-packages/sympy/__init__.py


In [ ]:
!pip install --force-reinstall "sympy>=1.13.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 21.1 MB/s eta 0:00:00
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [ ]:
from transformers import MBartForConditionalGeneration
import torch

text_only_model = MBartForConditionalGeneration.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


text_only_model = text_only_model.to(device)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

In [ ]:
class TextOnlyDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        source_column="source_text",
        target_column="target_text",
        max_source_length=128,
        max_target_length=128
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer

        self.source_column = source_column
        self.target_column = target_column

        self.max_source_length = max_source_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        source_text = str(row[self.source_column])
        target_text = str(row[self.target_column])

        source = self.tokenizer(
            source_text,
            max_length=self.max_source_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        target = self.tokenizer(
            target_text,
            max_length=self.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = target["input_ids"].squeeze(0)

        # Ignore padding tokens when calculating loss
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": source["input_ids"].squeeze(0),
            "attention_mask": source["attention_mask"].squeeze(0),
            "labels": labels
        }

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from pathlib import Path

from transformers import (
    MBart50TokenizerFast,
    MBartForConditionalGeneration
)

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Indic-Multimodal-NMT"
)

text_only_model = MBartForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

CHECKPOINT_FILE = PROJECT_ROOT / "checkpoints/translation_checkpoint.csv"
df_train = pd.read_csv(
    CHECKPOINT_FILE
)

text_dataset = TextOnlyDataset(
    dataframe=df_train,
    tokenizer=tokenizer,
    source_column="source_text",
    target_column="target_text"
)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

In [ ]:
text_loader = DataLoader(
    text_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

print("Dataset:", len(text_dataset))
print("Batches:", len(text_loader))

Dataset: 25840
Batches: 25840


In [ ]:
print(df_train.columns.tolist())


['image_id', 'filename', 'source_text', 'target_text']


In [ ]:

for param in text_only_model.model.encoder.parameters():
    param.requires_grad = False

text_only_model.train()

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        text_only_model.parameters()
    ),
    lr=1e-5
) #

MAX_STEPS = 10
for step, batch in enumerate(text_loader):

    if step >= MAX_STEPS:       # CPU sanity test
        break

    batch = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }

    optimizer.zero_grad(set_to_none=True)

    outputs = text_only_model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"]
    )

    loss = outputs.loss

    loss.backward()
    optimizer.step()

    print(
        f"Step {step + 1}/{MAX_STEPS} | "
        f"Loss: {loss.item():.4f}"
    )

Step 1/10 | Loss: 1.7696
Step 2/10 | Loss: 2.1527
Step 3/10 | Loss: 1.2835
Step 4/10 | Loss: 1.2517
Step 5/10 | Loss: 1.4232
Step 6/10 | Loss: 1.5984
Step 7/10 | Loss: 1.8686
Step 8/10 | Loss: 1.9900
Step 9/10 | Loss: 1.0889
Step 10/10 | Loss: 1.4570


In [ ]:
pip install sacrebleu

In [ ]:
# =========================
# BLEU ON TEST SET
# =========================

text_only_model.eval()

predictions = []
references = []

with torch.no_grad():

    for step, batch in enumerate(text_loader):

        batch = {
            k: v.to(device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        generated_ids = text_only_model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=128,
            num_beams=2
        )

        preds = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        labels = batch["labels"].clone()
        labels[labels == -100] = tokenizer.pad_token_id

        refs = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True
        )

        predictions.extend(preds)
        references.extend(refs)


bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

print(f"\nBLEU after 10 training steps: {bleu.score:.2f}")

KeyboardInterrupt: 

In [ ]:
import sacrebleu
text_only_model.eval()

predictions = []
references = []

with torch.no_grad():

    for step, batch in enumerate(text_loader):

        if step >= 20:
           break

        batch = {
            k: v.to(device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        generated_ids = text_only_model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=64,
            num_beams=1,       # much faster
        )

        preds = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        labels = batch["labels"].clone()
        labels[labels == -100] = tokenizer.pad_token_id

        refs = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True
        )

        predictions.extend(preds)
        references.extend(refs)

bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

print(f"BLEU: {bleu.score:.2f}")

BLEU: 2.67
